# L14a: Linear State Space Models and Stability
In this lecture, we solve linear time-invariant state space models, in which the state changes at a rate set by the current state and an input. We read the stability of a model from the eigenvalues of its state matrix. Then we linearize a nonlinear model about an operating point.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
>
> * __Solve the open-loop model and read its stability from the eigenvalues:__ Without an input, the solution of a linear state space model is a sum of modes, one for each eigenvalue of the state matrix. We'll see how the real part of each eigenvalue decides whether its mode grows or decays, and how the imaginary part sets its oscillation.
> * __Solve the forced model one time step at a time:__ With an input, the solution contains an integral of the input over time, which has no closed form for a general input. We'll hold the input constant over each step and derive two discretizations, the bilinear discretization and the zero-order hold.
> * __Linearize a nonlinear model about an operating point:__ Near a chosen state and input, a first-order Taylor expansion approximates a nonlinear model with a linear state space model whose state and input matrices are Jacobians. We'll linearize about a steady state and read the stability of that steady state from the eigenvalues of the state matrix.

Let's get started!

___

## Examples
We will use the following example to illustrate the key ideas of this lecture:

> [▶ Linearize a gene circuit](CHEME-5800-L14a-Example-GeneCircuits-Fall-2026.ipynb). In this example, we build the model of a gene circuit from its config file, solve it with an ODE solver, and find its steady state and eigenvalues. We linearize the model about the steady state, and solve the linear model with the bilinear or the zero-order hold discretization. You can change the circuit, the pulse of an added chemical, and the config file, and rerun to see where the linear model works and where it fails.

The [advanced eigenvalue notebook](CHEME-5800-L14a-Advanced-ComputingEigenvalues-Fall-2026.ipynb) shows how numerical methods compute eigenvalues and eigenvectors, with power iteration and QR iteration.

[Stability of the bilinear discretization](CHEME-5800-L14a-Advanced-Derivation-BilinearStability-Fall-2026.ipynb). In this notebook, we show why the bilinear discretization keeps the stability of the continuous model, for any time step. It is a derivation notebook with no code.

___

## Linear State Space Models
A linear time-invariant (LTI) state space model describes how a vector of states changes over time in response to an input.

> __Continuous-time LTI state space model:__
>
> Let $\mathbf{x}(t)\in\mathbb{R}^{n}$ be the vector of $n$ states and $\mathbf{u}(t)\in\mathbb{R}^{d}$ the vector of $d$ inputs at time $t$. The model is given by:
>
> $$
> \boxed{\dot{\mathbf{x}}(t) = \mathbf{A}\mathbf{x}(t)+\mathbf{B}\mathbf{u}(t),\qquad \mathbf{x}(0)=\mathbf{x}_{0}}
> $$
>
> where $\dot{\mathbf{x}}$ is the time derivative of the state, $\mathbf{A}\in\mathbb{R}^{n\times n}$ is the __state matrix__, $\mathbf{B}\in\mathbb{R}^{n\times d}$ is the __input matrix__, and $\mathbf{x}_{0}\in\mathbb{R}^{n}$ is the initial state. The model is __time-invariant__ because $\mathbf{A}$ and $\mathbf{B}$ do not depend on $t$.

The state collects the quantities that describe the system at time $t$, such as the concentrations in a reactor. The input collects what we set from outside, such as a feed rate. Because the model is linear and its matrices are constant, we can write its solution exactly.

We first consider the __open-loop__ case, in which the input is zero, $\mathbf{u}=\mathbf{0}$, and the model reduces to $\dot{\mathbf{x}}=\mathbf{A}\mathbf{x}$. Its solution is given by the __matrix exponential__:

$$
\mathbf{x}(t) = e^{\mathbf{A}t}\mathbf{x}_{0},\qquad
e^{\mathbf{A}t} = \mathbf{I}+\mathbf{A}t+\frac{(\mathbf{A}t)^{2}}{2!}+\frac{(\mathbf{A}t)^{3}}{3!}+\dots
$$

where $\mathbf{I}$ is the $n\times n$ identity matrix. Differentiating the series term by term gives $\frac{d}{dt}e^{\mathbf{A}t}=\mathbf{A}e^{\mathbf{A}t}$, which shows that this solution satisfies the model. At $t=0$, the series gives $e^{\mathbf{0}}=\mathbf{I}$ and the initial state $\mathbf{x}_{0}$.

### Modes and the Eigendecomposition
An __eigenvector__ $\mathbf{v}$ of $\mathbf{A}$ is a nonzero vector that $\mathbf{A}$ only stretches, $\mathbf{A}\mathbf{v}=\lambda\mathbf{v}$, and the stretch factor $\lambda$ is its __eigenvalue__. Both can be complex, $\mathbf{v}\in\mathbb{C}^{n}$ and $\lambda\in\mathbb{C}$, where $\mathbb{C}$ is the set of complex numbers. The [advanced eigenvalue notebook](CHEME-5800-L14a-Advanced-ComputingEigenvalues-Fall-2026.ipynb) computes eigenvalues and eigenvectors with power iteration and QR iteration.

When $\mathbf{A}$ has $n$ linearly independent eigenvectors, its eigendecomposition is $\mathbf{A}=\mathbf{V}\boldsymbol{\Lambda}\mathbf{V}^{-1}$, where the columns of the $n\times n$ matrix $\mathbf{V}$ are the eigenvectors $\mathbf{v}_{1},\dots,\mathbf{v}_{n}$ and the $n\times n$ diagonal matrix $\boldsymbol{\Lambda}$ holds the eigenvalues $\lambda_{1},\dots,\lambda_{n}$. Every power of the state matrix is then $\mathbf{A}^{k}=\mathbf{V}\boldsymbol{\Lambda}^{k}\mathbf{V}^{-1}$. Substituting these powers into the series gives:

$$
\begin{aligned}
\mathbf{x}(t) &= \mathbf{V}e^{\boldsymbol{\Lambda}t}\mathbf{V}^{-1}\mathbf{x}_{0}\\
&= c_{1}e^{\lambda_{1}t}\mathbf{v}_{1}+c_{2}e^{\lambda_{2}t}\mathbf{v}_{2}+\dots+c_{n}e^{\lambda_{n}t}\mathbf{v}_{n}
\end{aligned}
$$

where $e^{\boldsymbol{\Lambda}t}=\operatorname{diag}(e^{\lambda_{1}t},\dots,e^{\lambda_{n}t})$, and the coefficients $c_{1},\dots,c_{n}$, the entries of $\mathbf{c}=\mathbf{V}^{-1}\mathbf{x}_{0}$, express the initial state in the eigenvector basis. Each term $c_{i}e^{\lambda_{i}t}\mathbf{v}_{i}$ is a __mode__ of the model. A mode keeps the direction of its eigenvector, and its size changes as $e^{\lambda_{i}t}$.

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  body[data-jp-theme-light="true"] .course-diagram { color-scheme: light; }
  body[data-jp-theme-light="false"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<p align="center">
    <img class="course-diagram" src="figs/Fig-Eigenvectors/Fig-Eigenvectors.svg" width="640" style="max-width:100%; height:auto;" alt="Two panels. Left, panel a: a matrix A with eigenvalues 2 and 0.5 acts on three vectors drawn from the origin. Along two dashed red lines, the eigenvector v1 is stretched to A v1 = 2 v1 and the eigenvector v2 is shrunk to A v2 = 0.5 v2, and each stays on its line. A general vector w, in black, maps to A w, in blue, which points in a different direction. Right, panel b: a stable matrix with the same eigenvectors and eigenvalues minus 0.5 and minus 2. The initial state x(0) is the sum of two red arrows along the eigenvector lines, c1 v1 and c2 v2, joined by a dotted parallelogram. A blue solution curve runs from x(0) toward the origin. At x(1), a second dotted parallelogram shows the two modes one time unit later: the v1 arrow has shrunk a little, the v2 arrow has almost vanished, and the curve approaches the origin along v1."/>
</p>

__Eigenvectors and modes.__ (a) The matrix $\mathbf{A}$ stretches each eigenvector along its own dashed line, while a general vector $\mathbf{w}$ changes direction. (b) For a stable matrix with the same eigenvectors, the initial state splits into the modes $c_{1}\mathbf{v}_{1}$ and $c_{2}\mathbf{v}_{2}$. By $t=1$, the fast mode has shrunk by the factor $e^{-2}$ and the slow mode by $e^{-0.5}$, which bends the solution toward $\mathbf{v}_{1}$.

### Stability from the Eigenvalues
A __steady state__ of the open-loop model is a state where $\dot{\mathbf{x}}=\mathbf{0}$, such as $\mathbf{x}=\mathbf{0}$. Write a complex eigenvalue as $\lambda=\alpha+i\beta$, with real part $\alpha$ and imaginary part $\beta$. Euler's formula then gives:

$$
e^{\lambda t} = \underbrace{e^{\alpha t}}_{\updownarrow}\,\cdot\,\overbrace{\left(\cos\beta t+i\sin\beta t\right)}^{\text{oscillation}}.
$$

The real part $\alpha$ sets the growth or decay of the mode, and the imaginary part $\beta$ sets its oscillation. A real eigenvalue has $\beta=0$, and its mode does not oscillate. Complex eigenvalues of a real matrix come in conjugate pairs, $\alpha\pm i\beta$, with conjugate eigenvectors. For a real initial state, the imaginary parts of the two modes of a pair cancel, and the pair gives a real oscillation.

> __Stability of the open-loop model:__
>
> The steady state $\mathbf{x}=\mathbf{0}$ of $\dot{\mathbf{x}}=\mathbf{A}\mathbf{x}$ is __stable__ when every eigenvalue of $\mathbf{A}$ has a negative real part, because then every mode decays to zero. It is __unstable__ when at least one eigenvalue has a positive real part, because that mode grows without bound.
>
> A zero eigenvalue gives a constant mode, and a purely imaginary pair gives an oscillation. For a model with two states, $n=2$, the eigenvalues are the roots of the quadratic $\lambda^{2}-\tau\lambda+\det(\mathbf{A})=0$, where the trace $\tau$ is the sum of the diagonal entries of $\mathbf{A}$ and $\det(\mathbf{A})$ is the determinant of $\mathbf{A}$:
>
> $$
> \lambda_{1,2} = \frac{\tau}{2}\pm\sqrt{\frac{\tau^{2}}{4}-\det(\mathbf{A})}.
> $$
> The eigenvalues are complex when $\tau^{2}/4<\det(\mathbf{A})$. Both real parts are negative exactly when $\tau<0$ and $\det(\mathbf{A})>0$.

### Phase Portraits
A __phase portrait__ draws the solution curves of a two-state model in the plane of its two states $x_{1}$ and $x_{2}$, one curve for each initial state.

<p align="center">
    <img class="course-diagram" src="figs/Fig-PhasePortraits/Fig-PhasePortraits.svg" width="640" style="max-width:100%; height:auto;" alt="Four phase portraits in a two by two grid, each on axes x1 and x2 from minus 2 to 2. Top left, stable node with eigenvalues minus 1 and minus 3: blue curves from the edges of the box run into the origin and approach it along the horizontal axis. Top right, saddle with eigenvalues 1 and minus 2: blue curves come in along the vertical axis and leave along the horizontal axis. Bottom left, center with eigenvalues plus and minus 2i: three blue circles around the origin, traveled clockwise. Bottom right, stable spiral with eigenvalues minus 0.3 plus and minus 2i: two blue curves spiral clockwise into the origin. In the top two panels, dashed red lines along the axes mark the eigenvector directions and carry their eigenvalues."/>
</p>

__Phase portraits.__ In a __stable node__ and a __stable spiral__, every curve runs into the origin, and the spiral turns on the way in. In a __saddle__, curves come in along the eigenvector of the negative eigenvalue and leave along the eigenvector of the positive eigenvalue. In a __center__, each curve stays on a closed loop around the origin.

___

## The Forced Solution and Discretization
Now suppose the input is not zero. We multiply the model by $e^{-\mathbf{A}t}$, which turns its left side into a single derivative. Because $e^{-\mathbf{A}t}$ is a series in powers of $\mathbf{A}$, it commutes with $\mathbf{A}$, and $\frac{d}{dt}e^{-\mathbf{A}t}=-\mathbf{A}e^{-\mathbf{A}t}=-e^{-\mathbf{A}t}\mathbf{A}$:

$$
\begin{aligned}
\dot{\mathbf{x}}(t) &= \mathbf{A}\mathbf{x}(t)+\mathbf{B}\mathbf{u}(t) &&\quad\text{(the forced model)}\\
\dot{\mathbf{x}}(t)-\mathbf{A}\mathbf{x}(t) &= \mathbf{B}\mathbf{u}(t) &&\quad\text{(move the state term to the left)}\\
\underbrace{e^{-\mathbf{A}t}\dot{\mathbf{x}}(t)-e^{-\mathbf{A}t}\mathbf{A}\mathbf{x}(t)}_{\displaystyle =\frac{d}{dt}\left(e^{-\mathbf{A}t}\mathbf{x}(t)\right)} &= e^{-\mathbf{A}t}\mathbf{B}\mathbf{u}(t) &&\quad\text{(multiply on the left by }e^{-\mathbf{A}t}\text{)}\\
\frac{d}{dt}\left(e^{-\mathbf{A}t}\mathbf{x}(t)\right) &= e^{-\mathbf{A}t}\mathbf{B}\mathbf{u}(t) &&\quad\text{(product rule, in reverse)}\\
\int_{0}^{t}\frac{d}{ds}\left(e^{-\mathbf{A}s}\mathbf{x}(s)\right)ds &= \int_{0}^{t}e^{-\mathbf{A}s}\mathbf{B}\mathbf{u}(s)\,ds &&\quad\text{(integrate both sides from }0\text{ to }t\text{)}\\
e^{-\mathbf{A}t}\mathbf{x}(t)-\mathbf{x}_{0} &= \int_{0}^{t}e^{-\mathbf{A}s}\mathbf{B}\mathbf{u}(s)\,ds &&\quad\text{(fundamental theorem of calculus, with }e^{\mathbf{0}}=\mathbf{I}\text{)}\\
\mathbf{x}(t) &= e^{\mathbf{A}t}\mathbf{x}_{0}+\int_{0}^{t}e^{\mathbf{A}(t-s)}\mathbf{B}\mathbf{u}(s)\,ds &&\quad\text{(multiply on the left by }e^{\mathbf{A}t}\text{)}
\end{aligned}
$$

In the last step, $e^{\mathbf{A}t}e^{-\mathbf{A}t}=\mathbf{I}$ and $e^{\mathbf{A}t}e^{-\mathbf{A}s}=e^{\mathbf{A}(t-s)}$. The first term of the result is the open-loop solution. The integral adds the effect of the input at every earlier time $s$, carried forward by the open-loop model over the remaining time $t-s$. For a general input, this integral has no closed form.

A __discretization__ advances the state from one time to the next on a grid of times $t_{k}=k\Delta t$, with a step $\Delta t$. We'll derive two discretizations, the bilinear discretization and the zero-order hold, which both hold the input constant at $\mathbf{u}_{k}\in\mathbb{R}^{d}$ over each step.

### The Bilinear Discretization
The __trapezoid rule__ approximates the integral of a function $g$ over one step by the step times the average of its two end values:

$$
\int_{t_{k-1}}^{t_{k}}g(s)\,ds\approx\frac{\Delta t}{2}\left(g(t_{k-1})+g(t_{k})\right).
$$

We start from the forced model, $\dot{\mathbf{x}}=\mathbf{A}\mathbf{x}+\mathbf{B}\mathbf{u}$, integrate both sides over one step, and apply this rule to the state term:

$$
\begin{aligned}
\int_{t_{k-1}}^{t_{k}}\dot{\mathbf{x}}(s)\,ds &= \int_{t_{k-1}}^{t_{k}}\left(\mathbf{A}\mathbf{x}(s)+\mathbf{B}\mathbf{u}(s)\right)ds &&\quad\text{(integrate both sides from }t_{k-1}\text{ to }t_{k}\text{)}\\
\mathbf{x}(t_{k})-\mathbf{x}(t_{k-1}) &= \int_{t_{k-1}}^{t_{k}}\left(\mathbf{A}\mathbf{x}(s)+\mathbf{B}\mathbf{u}_{k}\right)ds &&\quad\text{(fundamental theorem of calculus; }\mathbf{u}(s)=\mathbf{u}_{k}\text{)}\\
&= \mathbf{A}\int_{t_{k-1}}^{t_{k}}\mathbf{x}(s)\,ds+\Delta t\,\mathbf{B}\mathbf{u}_{k} &&\quad\text{(}\mathbf{A}\text{ and }\mathbf{B}\mathbf{u}_{k}\text{ are constant over the step)}\\
&\approx \frac{\Delta t}{2}\mathbf{A}\left(\mathbf{x}(t_{k-1})+\mathbf{x}(t_{k})\right)+\Delta t\,\mathbf{B}\mathbf{u}_{k} &&\quad\text{(trapezoid rule)}\\
\mathbf{x}_{k}-\mathbf{x}_{k-1} &= \frac{\Delta t}{2}\mathbf{A}\left(\mathbf{x}_{k-1}+\mathbf{x}_{k}\right)+\Delta t\,\mathbf{B}\mathbf{u}_{k} &&\quad\text{(replace each }\mathbf{x}(t_{k})\text{ by its approximation }\mathbf{x}_{k}\text{)}\\
\mathbf{x}_{k}-\frac{\Delta t}{2}\mathbf{A}\mathbf{x}_{k} &= \mathbf{x}_{k-1}+\frac{\Delta t}{2}\mathbf{A}\mathbf{x}_{k-1}+\Delta t\,\mathbf{B}\mathbf{u}_{k} &&\quad\text{(collect the }\mathbf{x}_{k}\text{ terms on the left)}\\
\left(\mathbf{I}-\tfrac{\Delta t}{2}\mathbf{A}\right)\mathbf{x}_{k} &= \left(\mathbf{I}+\tfrac{\Delta t}{2}\mathbf{A}\right)\mathbf{x}_{k-1}+\Delta t\,\mathbf{B}\mathbf{u}_{k} &&\quad\text{(factor, using }\mathbf{x}=\mathbf{I}\mathbf{x}\text{)}\\
\mathbf{x}_{k} &= \underbrace{\left(\mathbf{I}-\tfrac{\Delta t}{2}\mathbf{A}\right)^{-1}\left(\mathbf{I}+\tfrac{\Delta t}{2}\mathbf{A}\right)}_{\bar{\mathbf{A}}}\,\mathbf{x}_{k-1}+\underbrace{\left(\mathbf{I}-\tfrac{\Delta t}{2}\mathbf{A}\right)^{-1}\Delta t\,\mathbf{B}}_{\bar{\mathbf{B}}}\,\mathbf{u}_{k} &&\quad\text{(multiply through by the inverse)}
\end{aligned}
$$

The last step needs the inverse of $\mathbf{I}-\tfrac{\Delta t}{2}\mathbf{A}$, which exists when no eigenvalue of $\mathbf{A}$ equals $2/\Delta t$. Every stable model meets this condition. The matrices $\bar{\mathbf{A}}$ and $\bar{\mathbf{B}}$ in the last line define the bilinear discretization:

> __Bilinear discretization:__
>
> Given the state matrix $\mathbf{A}$, the input matrix $\mathbf{B}$, and a step $\Delta t>0$, the discrete matrices $\bar{\mathbf{A}}\in\mathbb{R}^{n\times n}$ and $\bar{\mathbf{B}}\in\mathbb{R}^{n\times d}$ are given by:
>
> $$
> \boxed{\bar{\mathbf{A}} = \left(\mathbf{I}-\tfrac{\Delta t}{2}\mathbf{A}\right)^{-1}\left(\mathbf{I}+\tfrac{\Delta t}{2}\mathbf{A}\right),\qquad
> \bar{\mathbf{B}} = \left(\mathbf{I}-\tfrac{\Delta t}{2}\mathbf{A}\right)^{-1}\Delta t\,\mathbf{B}}
> $$
>
> where $\mathbf{I}$ is the $n\times n$ identity matrix. The state follows the recursion, starting from $\mathbf{x}_{0}$:
>
> $$
> \mathbf{x}_{k} = \bar{\mathbf{A}}\mathbf{x}_{k-1}+\bar{\mathbf{B}}\mathbf{u}_{k},\qquad k = 1,2,\dots
> $$
> 
> The recursion keeps the stability of the continuous model. Each eigenvector $\mathbf{v}$ of $\mathbf{A}$ is also an eigenvector of $\bar{\mathbf{A}}$, but its eigenvalue changes from $\lambda$ to $\bar{\lambda}$:
>
> $$
> \bar{\lambda}=\frac{1+\Delta t\,\lambda/2}{1-\Delta t\,\lambda/2},\qquad |\bar{\lambda}|<1\iff\mathrm{Re}(\lambda)<0\quad\text{(stable model)}.
> $$
>
> Each mode of the recursion changes by the factor $\bar{\lambda}$ in each step, and it decays when $|\bar{\lambda}|<1$. A stable model therefore gives a stable recursion, for any step $\Delta t$. The [companion derivation](CHEME-5800-L14a-Advanced-Derivation-BilinearStability-Fall-2026.ipynb) proves both results.

The trapezoid rule makes the recursion second-order accurate, which means its error over a fixed time scales with $\Delta t^{2}$. Halving the step cuts the error by about a factor of four.

### The Zero-Order Hold Discretization
The bilinear discretization approximates the integral of the state over each step with the trapezoid rule. When the input is constant over each step, the forced solution gives the state at the end of each step exactly. This exact discretization is the __zero-order hold__, named for the constant input, a polynomial of order zero in time.

We start from the forced model and integrate it over one step, from $t_{k-1}$ to $t_{k}$:

$$
\begin{aligned}
\dot{\mathbf{x}}(t) &= \mathbf{A}\mathbf{x}(t)+\mathbf{B}\mathbf{u}(t) &&\quad\text{(the forced model)}\\
\dot{\mathbf{x}}(t)-\mathbf{A}\mathbf{x}(t) &= \mathbf{B}\mathbf{u}(t) &&\quad\text{(move the state term to the left)}\\
\underbrace{e^{-\mathbf{A}t}\dot{\mathbf{x}}(t)-e^{-\mathbf{A}t}\mathbf{A}\mathbf{x}(t)}_{\displaystyle =\frac{d}{dt}\left(e^{-\mathbf{A}t}\mathbf{x}(t)\right)} &= e^{-\mathbf{A}t}\mathbf{B}\mathbf{u}(t) &&\quad\text{(multiply on the left by }e^{-\mathbf{A}t}\text{)}\\
\frac{d}{dt}\left(e^{-\mathbf{A}t}\mathbf{x}(t)\right) &= e^{-\mathbf{A}t}\mathbf{B}\mathbf{u}(t) &&\quad\text{(product rule, in reverse, with }\tfrac{d}{dt}e^{-\mathbf{A}t}=-e^{-\mathbf{A}t}\mathbf{A}\text{)}\\
\int_{t_{k-1}}^{t_{k}}\frac{d}{ds}\left(e^{-\mathbf{A}s}\mathbf{x}(s)\right)ds &= \int_{t_{k-1}}^{t_{k}}e^{-\mathbf{A}s}\mathbf{B}\mathbf{u}(s)\,ds &&\quad\text{(integrate both sides from }t_{k-1}\text{ to }t_{k}\text{, with }s\text{ the time inside the step)}\\
e^{-\mathbf{A}t_{k}}\mathbf{x}(t_{k})-e^{-\mathbf{A}t_{k-1}}\mathbf{x}(t_{k-1}) &= \int_{t_{k-1}}^{t_{k}}e^{-\mathbf{A}s}\mathbf{B}\mathbf{u}(s)\,ds &&\quad\text{(fundamental theorem of calculus)}\\
e^{-\mathbf{A}t_{k}}\mathbf{x}(t_{k}) &= e^{-\mathbf{A}t_{k-1}}\mathbf{x}(t_{k-1})+\int_{t_{k-1}}^{t_{k}}e^{-\mathbf{A}s}\mathbf{B}\mathbf{u}(s)\,ds &&\quad\text{(move the }\mathbf{x}(t_{k-1})\text{ term to the right)}\\
\mathbf{x}(t_{k}) &= e^{\mathbf{A}t_{k}}e^{-\mathbf{A}t_{k-1}}\mathbf{x}(t_{k-1})+e^{\mathbf{A}t_{k}}\int_{t_{k-1}}^{t_{k}}e^{-\mathbf{A}s}\mathbf{B}\mathbf{u}(s)\,ds &&\quad\text{(multiply on the left by }e^{\mathbf{A}t_{k}}\text{, with }e^{\mathbf{A}t_{k}}e^{-\mathbf{A}t_{k}}=\mathbf{I}\text{)}\\
&= e^{\mathbf{A}\Delta t}\mathbf{x}(t_{k-1})+e^{\mathbf{A}t_{k}}\int_{t_{k-1}}^{t_{k}}e^{-\mathbf{A}s}\mathbf{B}\mathbf{u}(s)\,ds &&\quad\text{(exponents of the same matrix add: }e^{\mathbf{A}t_{k}}e^{-\mathbf{A}t_{k-1}}=e^{\mathbf{A}(t_{k}-t_{k-1})}=e^{\mathbf{A}\Delta t}\text{)}\\
&= e^{\mathbf{A}\Delta t}\mathbf{x}(t_{k-1})+\int_{t_{k-1}}^{t_{k}}e^{\mathbf{A}(t_{k}-s)}\mathbf{B}\mathbf{u}(s)\,ds &&\quad\text{(}e^{\mathbf{A}t_{k}}\text{ does not depend on }s\text{; inside the integral, }e^{\mathbf{A}t_{k}}e^{-\mathbf{A}s}=e^{\mathbf{A}(t_{k}-s)}\text{)}
\end{aligned}
$$

This result is exact for any input. The first term is the open-loop solution over one step, starting from $\mathbf{x}(t_{k-1})$. The integral adds the input at each time $s$ inside the step, carried forward by the open-loop model over the remaining time $t_{k}-s$.

Now we hold the input at $\mathbf{u}_{k}$ over the step, and evaluate the integral:

$$
\begin{aligned}
\mathbf{x}(t_{k}) &= e^{\mathbf{A}\Delta t}\mathbf{x}(t_{k-1})+\left(\int_{t_{k-1}}^{t_{k}}e^{\mathbf{A}(t_{k}-s)}\,ds\right)\mathbf{B}\mathbf{u}_{k} &&\quad\text{(}\mathbf{u}(s)=\mathbf{u}_{k}\text{, and the constant }\mathbf{B}\mathbf{u}_{k}\text{ moves out of the integral)}\\
&= e^{\mathbf{A}\Delta t}\mathbf{x}(t_{k-1})-\left(\int_{\Delta t}^{0}e^{\mathbf{A}\sigma}\,d\sigma\right)\mathbf{B}\mathbf{u}_{k} &&\quad\text{(substitute }\sigma=t_{k}-s\text{: }ds=-d\sigma\text{, and }s=t_{k-1},\,t_{k}\text{ give }\sigma=\Delta t,\,0\text{)}\\
&= e^{\mathbf{A}\Delta t}\mathbf{x}(t_{k-1})+\left(\int_{0}^{\Delta t}e^{\mathbf{A}\sigma}\,d\sigma\right)\mathbf{B}\mathbf{u}_{k} &&\quad\text{(swap the limits, which removes the minus sign)}\\
&= e^{\mathbf{A}\Delta t}\mathbf{x}(t_{k-1})+\mathbf{A}^{-1}\left(e^{\mathbf{A}\Delta t}-e^{\mathbf{0}}\right)\mathbf{B}\mathbf{u}_{k} &&\quad\text{(the antiderivative of }e^{\mathbf{A}\sigma}\text{ is }\mathbf{A}^{-1}e^{\mathbf{A}\sigma}\text{, since }\tfrac{d}{d\sigma}e^{\mathbf{A}\sigma}=\mathbf{A}e^{\mathbf{A}\sigma}\text{)}\\
\mathbf{x}_{k} &= \underbrace{e^{\mathbf{A}\Delta t}}_{\bar{\mathbf{A}}}\,\mathbf{x}_{k-1}+\underbrace{\mathbf{A}^{-1}\left(e^{\mathbf{A}\Delta t}-\mathbf{I}\right)\mathbf{B}}_{\bar{\mathbf{B}}}\,\mathbf{u}_{k} &&\quad\text{(}e^{\mathbf{0}}=\mathbf{I}\text{; write }\mathbf{x}_{k}\text{ for }\mathbf{x}(t_{k})\text{)}
\end{aligned}
$$

No step above is an approximation. When the input is constant over each step, $\mathbf{x}_{k}$ equals the state $\mathbf{x}(t_{k})$ of the continuous model at every grid time, for any step $\Delta t$.

The antiderivative $\mathbf{A}^{-1}e^{\mathbf{A}\sigma}$ needs the inverse of $\mathbf{A}$, which exists when no eigenvalue of $\mathbf{A}$ is zero. Every stable model meets this condition. The matrices $\bar{\mathbf{A}}$ and $\bar{\mathbf{B}}$ in the last line define the zero-order hold discretization:

> __Zero-order hold discretization:__
>
> Given the state matrix $\mathbf{A}$, the input matrix $\mathbf{B}$, and a step $\Delta t>0$, the discrete matrices $\bar{\mathbf{A}}\in\mathbb{R}^{n\times n}$ and $\bar{\mathbf{B}}\in\mathbb{R}^{n\times d}$ are given by:
>
> $$
> \boxed{\bar{\mathbf{A}} = e^{\mathbf{A}\Delta t},\qquad
> \bar{\mathbf{B}} = \mathbf{A}^{-1}\left(e^{\mathbf{A}\Delta t}-\mathbf{I}\right)\mathbf{B}}
> $$
>
> The state follows the same recursion as in the bilinear discretization, $\mathbf{x}_{k}=\bar{\mathbf{A}}\mathbf{x}_{k-1}+\bar{\mathbf{B}}\mathbf{u}_{k}$, starting from $\mathbf{x}_{0}$. Each eigenvector $\mathbf{v}$ of $\mathbf{A}$ is also an eigenvector of $\bar{\mathbf{A}}$, but its eigenvalue changes from $\lambda$ to $\bar{\lambda}$:
>
> $$
> \bar{\lambda}=e^{\lambda\Delta t},\qquad |\bar{\lambda}|=e^{\mathrm{Re}(\lambda)\Delta t}<1\iff\mathrm{Re}(\lambda)<0\quad\text{(stable model)}.
> $$
>
> Each power $\mathbf{A}^{j}$ in the series for $e^{\mathbf{A}\Delta t}$ multiplies $\mathbf{v}$ by $\lambda^{j}$, which gives $\bar{\lambda}=e^{\lambda\Delta t}$. Euler's formula splits $\bar{\lambda}$ into the real factor $e^{\mathrm{Re}(\lambda)\Delta t}$ and an oscillation factor of size one. A stable model therefore gives a stable recursion, for any step $\Delta t$.

The zero-order hold needs the matrix exponential $e^{\mathbf{A}\Delta t}$. The bilinear discretization needs only the inverse of $\mathbf{I}-\tfrac{\Delta t}{2}\mathbf{A}$.

___

## Linearization
Most models of real systems are nonlinear, $\dot{\mathbf{x}}=\mathbf{f}(\mathbf{x},\mathbf{u})$, where the function $\mathbf{f}$ gives the rate of change of each state, $\dot{x}_{i}=f_{i}(\mathbf{x},\mathbf{u})$ for $i=1,\dots,n$. We can approximate such a model near any chosen state $\mathbf{x}_{\text{op}}\in\mathbb{R}^{n}$ and input $\mathbf{u}_{\text{op}}\in\mathbb{R}^{d}$, called an __operating point__ $(\mathbf{x}_{\text{op}},\mathbf{u}_{\text{op}})$. Near this point, we write the state and the input as deviations, $\Delta\mathbf{x}=\mathbf{x}-\mathbf{x}_{\text{op}}$ and $\Delta\mathbf{u}=\mathbf{u}-\mathbf{u}_{\text{op}}$.

Because $\mathbf{x}_{\text{op}}$ is constant, $\dot{\mathbf{x}}$ is also the rate of change of $\Delta\mathbf{x}$. A first-order Taylor expansion of $\mathbf{f}$ about the operating point gives:

$$
\dot{\mathbf{x}} = \mathbf{f}(\mathbf{x}_{\text{op}}+\Delta\mathbf{x},\,\mathbf{u}_{\text{op}}+\Delta\mathbf{u})
\approx \mathbf{f}(\mathbf{x}_{\text{op}},\mathbf{u}_{\text{op}})+\mathbf{A}\,\Delta\mathbf{x}+\mathbf{B}\,\Delta\mathbf{u}.
$$

> __Linearization about an operating point:__
>
> Near the operating point $(\mathbf{x}_{\text{op}},\mathbf{u}_{\text{op}})$, the deviations follow the model:
>
> $$
> \boxed{\dot{\mathbf{x}} \approx \mathbf{A}\,\Delta\mathbf{x}+\mathbf{B}\,\Delta\mathbf{u}+\mathbf{f}_{\text{op}},\qquad
> a_{ij}=\left.\frac{\partial f_{i}}{\partial x_{j}}\right|_{(\mathbf{x}_{\text{op}},\mathbf{u}_{\text{op}})},\qquad
> b_{ij}=\left.\frac{\partial f_{i}}{\partial u_{j}}\right|_{(\mathbf{x}_{\text{op}},\mathbf{u}_{\text{op}})},\qquad
> \mathbf{f}_{\text{op}}=\mathbf{f}(\mathbf{x}_{\text{op}},\mathbf{u}_{\text{op}})}
> $$
>
> The state matrix $\mathbf{A}\in\mathbb{R}^{n\times n}$ is the __Jacobian__ of $\mathbf{f}$ with respect to the state, and $\mathbf{B}\in\mathbb{R}^{n\times d}$ is the Jacobian with respect to the input, both evaluated at the operating point. The constant vector $\mathbf{f}_{\text{op}}\in\mathbb{R}^{n}$ is the rate of change of the state at the operating point. 
> 
> The constant $\mathbf{f}_{\text{op}}$ acts like an input that never changes. With the input $(\Delta\mathbf{u},1)\in\mathbb{R}^{d+1}$ and the input matrix $[\,\mathbf{B}\;\;\mathbf{f}_{\text{op}}\,]\in\mathbb{R}^{n\times(d+1)}$, the deviations follow the forced model, and both discretizations apply unchanged.

The linear model is accurate only while the state and the input stay near the operating point. When the operating point is not a steady state, $\mathbf{f}_{\text{op}}\neq\mathbf{0}$ sets the state in motion. This drift can carry the state away from the point and limit how long the approximation stays accurate.

### Linearization about a Steady State
A __steady state__ $(\mathbf{x}^{\star},\mathbf{u}^{\star})$ of the nonlinear model satisfies $\mathbf{f}(\mathbf{x}^{\star},\mathbf{u}^{\star})=\mathbf{0}$. Steady states are the usual operating points, because the constant term vanishes there, $\mathbf{f}_{\text{op}}=\mathbf{0}$. The deviations then follow the linear model $\dot{\mathbf{x}}\approx\mathbf{A}\,\Delta\mathbf{x}+\mathbf{B}\,\Delta\mathbf{u}$.

With the input held at $\mathbf{u}^{\star}$, the eigenvalues of $\mathbf{A}$ decide whether small deviations decay or grow. When every eigenvalue has a negative real part, small deviations decay back to the steady state, which is then stable. When one eigenvalue has a positive real part, a small deviation grows, and the steady state is unstable. An eigenvalue with zero real part leaves the question open. 

Let's look at an example:

> __Example__
> 
> [▶ Linearize a gene circuit](CHEME-5800-L14a-Example-GeneCircuits-Fall-2026.ipynb). In this example, we build the model of a gene circuit from its config file, solve it with an ODE solver, and find its steady state and eigenvalues. We linearize the model about the steady state, and solve the linear model with the bilinear or the zero-order hold discretization. You can change the circuit, the pulse of an added chemical, and the config file, and rerun to see where the linear model works and where it fails.

___

## Summary
In this lecture, we solved linear time-invariant state space models with and without an input, and linearized a nonlinear model about an operating point.

> __Key Takeaways:__
>
> * __Eigenvalues decide stability:__ We wrote the open-loop solution as a sum of modes, one for each eigenvalue of the state matrix, and each mode grows or decays with the real part of its eigenvalue. The model is stable when every eigenvalue has a negative real part, and a complex pair of eigenvalues adds an oscillation.
>
> * __Two discretizations of the forced model:__ We held the input constant over each step, then derived the bilinear discretization from the trapezoid rule and the zero-order hold from the exact forced solution. Both recursions keep the stability of the continuous model for any step, and the zero-order hold gives the exact state at every grid time.
>
> * __Linearization gives a local linear model:__ We expanded a nonlinear model in a first-order Taylor series about an operating point, which gave the state and input matrices as Jacobians at that point. At a steady state, the expansion has no constant term, and with the input held fixed, small deviations decay when every eigenvalue of the state matrix has a negative real part.

Together, these ideas let us predict how a nonlinear model responds to small changes in its state and input near an operating point. A discretization of the linear model approximates that response one time step at a time.

___